# Laboratório de Ciência de Dados
## Aula 8 — Interpretabilidade de modelos utilizando SHAP e LIME

Este notebook é **autoinstrucional**. Leia as explicações e execute as células na ordem.

### Continuidade das aulas anteriores

Nas aulas anteriores, construímos, avaliamos e visualizamos diferentes modelos de aprendizado de máquina.

Até aqui, perguntas como estas foram centrais:

- o modelo acerta?
- qual é sua precisão?
- qual é seu recall?
- ele generaliza para novos dados?

Agora surge outra pergunta:

> **Por que o modelo tomou determinada decisão?**

Modelos como Random Forest, Gradient Boosting e redes neurais podem apresentar bom desempenho, mas sua lógica interna não é tão simples de comunicar quanto a de uma pequena Árvore de Decisão.

Nesta aula trabalharemos com **interpretabilidade de modelos** e **XAI — Explainable Artificial Intelligence**.

Utilizaremos duas técnicas:

- **SHAP** — SHapley Additive exPlanations;
- **LIME** — Local Interpretable Model-agnostic Explanations.

### Objetivos de aprendizagem

Ao final desta atividade, você deverá ser capaz de:

- explicar o que significa interpretabilidade em aprendizado de máquina;
- diferenciar interpretabilidade **global** e **local**;
- compreender a lógica conceitual do SHAP;
- compreender a lógica conceitual do LIME;
- utilizar SHAP para analisar a importância global dos atributos;
- utilizar SHAP para explicar uma previsão individual;
- utilizar LIME para explicar uma previsão individual;
- comparar explicações produzidas por SHAP e LIME;
- reconhecer limitações e instabilidades das explicações;
- evitar interpretar explicação preditiva como causalidade.

> Explicar um modelo não significa provar que ele está correto. A explicação mostra **como o modelo utilizou os dados para produzir sua saída**.

# 1. O que é interpretabilidade de modelos?

Em modelos simples, às vezes conseguimos acompanhar diretamente a lógica usada para produzir uma decisão.

Em uma pequena Árvore de Decisão, por exemplo, podemos seguir um caminho como:

```text
score <= 600?
    ↓
atrasos > 2?
    ↓
classe = alto risco
```

Em modelos formados por centenas de árvores ou estruturas ainda mais complexas, essa inspeção direta se torna difícil.

Chamamos de **interpretabilidade** a capacidade de compreender aspectos relevantes do comportamento do modelo.

A área de **XAI — Explainable Artificial Intelligence** reúne métodos que procuram tornar decisões algorítmicas mais compreensíveis para pessoas.

Isso pode ser útil para:

- investigar decisões inesperadas;
- compreender quais atributos estão influenciando previsões;
- analisar erros;
- comunicar resultados;
- auditar modelos;
- identificar possíveis problemas nos dados ou no comportamento aprendido.

Mas há um cuidado fundamental:

> **interpretabilidade não é sinônimo de correção, justiça ou causalidade.**

Um modelo pode ser perfeitamente explicável e continuar sendo ruim, enviesado ou baseado em dados inadequados.

# 2. Interpretabilidade global e local

Existem duas perguntas diferentes.

## Interpretabilidade global

Procura compreender o comportamento **geral** do modelo.

Exemplos:

- quais atributos mais influenciam as previsões?
- que variáveis costumam aumentar ou diminuir a saída do modelo?
- quais padrões aparecem repetidamente em muitas observações?

Uma explicação global tenta responder:

> **Como o modelo se comporta, em geral?**

## Interpretabilidade local

Procura compreender **uma previsão específica**.

Por exemplo:

> Por que esta transação recebeu probabilidade de fraude de 82%?

Uma explicação local tenta responder:

> **Por que o modelo produziu esta previsão para esta observação?**

Essa distinção é importante.

Um atributo pode ser muito importante globalmente e ter pouca influência em um caso específico. O contrário também pode ocorrer.

# 3. O que é SHAP?

**SHAP** significa **SHapley Additive exPlanations**.

O método é inspirado nos **valores de Shapley**, originalmente desenvolvidos na teoria dos jogos cooperativos.

Imagine que os atributos do modelo sejam jogadores que colaboram para produzir um resultado.

No nosso exemplo:

- `valor_transacao`;
- `dispositivo_novo`;
- `historico_chargeback`;
- `distancia_habitual_km`;
- etc.

O SHAP procura distribuir entre esses atributos a diferença entre uma **saída de referência** e a saída produzida para uma observação.

Uma representação simplificada é:

$$
f(x) =
\text{valor de referência}
+
\sum_{j=1}^{p}\phi_j
$$

onde:

- $f(x)$ é a saída do modelo para a observação $x$;
- o **valor de referência** representa a saída esperada do modelo para uma população de referência;
- $\phi_j$ é o valor SHAP do atributo $j$;
- $p$ é o número de atributos.

Para a classe **fraude**:

- valor SHAP **positivo** → o atributo empurra a saída em direção a maior risco de fraude;
- valor SHAP **negativo** → o atributo empurra a saída em direção a menor risco de fraude;
- valor próximo de zero → pouca contribuição naquela previsão.

## SHAP local e global

O SHAP produz explicações para observações individuais. A partir delas, podemos construir uma visão global.

**Local:** observamos os valores SHAP de uma transação específica.

**Global:** agregamos os valores de muitas transações, por exemplo calculando a média de $|\phi_j|$.

Nesta aula usaremos principalmente:

- **bar plot** → importância global;
- **beeswarm** → importância + direção + distribuição;
- **waterfall** → explicação de uma observação individual;
- **scatter** → relação entre o valor de um atributo e sua contribuição.

> SHAP explica a contribuição dos atributos **dentro do modelo**. Ele não demonstra que esses atributos causam o fenômeno observado.

# 4. O que é LIME?

**LIME** significa **Local Interpretable Model-agnostic Explanations**.

A ideia central é diferente da utilizada pelo SHAP.

O LIME não tenta explicar todo o modelo complexo diretamente. Para explicar uma observação específica, ele constrói uma **aproximação simples na vizinhança dessa observação**.

De forma simplificada:

1. escolhe a observação que queremos explicar;
2. cria várias observações artificiais próximas a ela;
3. envia essas observações ao modelo original;
4. observa as previsões produzidas;
5. atribui maior peso às observações mais próximas do caso original;
6. ajusta um modelo simples, normalmente linear, nessa vizinhança;
7. utiliza os coeficientes desse modelo simples como explicação local.

A lógica pode ser resumida como:

```text
modelo complexo
      ↓
observações próximas de um caso
      ↓
previsões do modelo complexo
      ↓
modelo simples local
      ↓
explicação
```

## Por que “model-agnostic”?

Porque o LIME não precisa conhecer a estrutura interna do algoritmo.

Ele precisa apenas conseguir enviar dados ao modelo e receber suas previsões.

Por isso pode, em princípio, explicar:

- Random Forest;
- Gradient Boosting;
- redes neurais;
- SVM;
- outros classificadores.

## Uma limitação importante

A explicação do LIME é **local**.

Ela descreve aproximadamente o comportamento do modelo perto daquela observação.

Não devemos generalizar automaticamente uma explicação LIME para toda a base.

Como o método utiliza perturbações e amostragem, as explicações também podem variar com:

- a semente aleatória;
- a definição da vizinhança;
- a discretização;
- o número de amostras.

Por isso, mais adiante analisaremos a **estabilidade** das explicações.

## SHAP e LIME não são a mesma coisa

| Aspecto | SHAP | LIME |
|---|---|---|
| Ideia principal | distribui a saída entre os atributos | aproxima o modelo localmente |
| Fundamento | valores de Shapley | modelo substituto local |
| Explicação local | sim | sim |
| Análise global | sim, agregando explicações | não é seu foco principal |
| Relação com o modelo | possui métodos específicos e agnósticos | agnóstico ao modelo |
| Resultado típico | contribuição de cada atributo | peso de condições/atributos locais |
| Estabilidade | depende do explicador e da referência | pode ser sensível à amostragem local |

Ao longo da aula explicaremos **a mesma transação com os dois métodos**.

Assim, será possível comparar não apenas os gráficos, mas também a lógica de cada abordagem.

# 5. Preparação do ambiente

Além das bibliotecas já utilizadas nas aulas anteriores, precisaremos de:

- `shap`;
- `lime`.

Esses pacotes não fazem parte do `scikit-learn`.

A célula abaixo verifica se estão instalados. Quando algum deles estiver ausente, o próprio notebook executará o equivalente a:

```python
%pip install shap lime
```

Depois da instalação, os pacotes são importados normalmente. A aula **não continuará silenciosamente sem SHAP ou LIME**.

In [ ]:
# Verifica dependências e instala automaticamente o que estiver ausente.
import importlib.util

pacotes_necessarios = {
    "shap": "shap",
    "lime": "lime"
}

pacotes_ausentes = [
    pacote_pip
    for modulo, pacote_pip in pacotes_necessarios.items()
    if importlib.util.find_spec(modulo) is None
]

if pacotes_ausentes:
    print(
        "Pacote(s) ausente(s):",
        ", ".join(pacotes_ausentes)
    )
    print("Instalando automaticamente...")

    # Executa o magic %pip do Jupyter.
    # É equivalente a digitar, por exemplo:
    # %pip install shap lime
    get_ipython().run_line_magic(
        "pip",
        "install -q " + " ".join(pacotes_ausentes)
    )

# Se a instalação falhar, os imports abaixo gerarão erro
# e interromperão a execução — não seguiremos sem as bibliotecas.
import shap
from lime.lime_tabular import LimeTabularExplainer

print("SHAP:", shap.__version__)

try:
    import lime
    print("LIME: instalado")
except Exception:
    print("LIME: importado por lime.lime_tabular")

In [ ]:
# Operações numéricas.
import numpy as np

# Manipulação de tabelas.
import pandas as pd

# Visualizações.
import matplotlib.pyplot as plt

# Separação treino/teste.
from sklearn.model_selection import train_test_split

# Modelo utilizado nesta aula.
from sklearn.ensemble import RandomForestClassifier

# Métricas apenas para verificar o comportamento do modelo.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# 6. Problema: detecção de transações suspeitas

Retomaremos o cenário da **VGAbank**.

O objetivo é classificar transações como:

```text
0 → legítima
1 → suspeita/fraude
```

Utilizaremos atributos com significado interpretável, porque queremos analisar **por que** o modelo considera determinadas transações mais arriscadas.

In [ ]:
# Número de transações.
n = 6000

# Valor da transação.
valor_transacao = np.clip(
    rng.lognormal(
        mean=5.4,
        sigma=0.9,
        size=n
    ),
    5,
    10000
)

# Quantidade de transações realizadas na última hora.
transacoes_1h = np.clip(
    rng.poisson(
        1.2,
        size=n
    ),
    0,
    10
)

# Distância, em km, do local habitual de compra.
distancia_habitual_km = np.clip(
    rng.exponential(
        scale=18,
        size=n
    ),
    0,
    400
)

# Idade da conta em dias.
idade_conta_dias = np.clip(
    rng.gamma(
        shape=3.0,
        scale=500,
        size=n
    ),
    10,
    5000
)

# Falhas de login nas últimas 24 horas.
falhas_login_24h = np.clip(
    rng.poisson(
        0.35,
        size=n
    ),
    0,
    8
)

# Variáveis binárias.
compra_internacional = rng.binomial(
    1,
    0.16,
    n
)

dispositivo_novo = rng.binomial(
    1,
    0.12,
    n
)

horario_noturno = rng.binomial(
    1,
    0.22,
    n
)

historico_chargeback = rng.binomial(
    1,
    0.07,
    n
)

# Percentual do limite utilizado.
limite_utilizado_pct = np.clip(
    rng.beta(
        2.2,
        2.8,
        n
    ),
    0,
    1
)

# -------------------------------------------------------
# Probabilidade didática de fraude
# -------------------------------------------------------
# A expressão abaixo serve apenas para gerar uma base
# sintética com relações interpretáveis.
logit = (
    -4.2
    + 0.00022 * valor_transacao
    + 0.48 * transacoes_1h
    + 0.012 * distancia_habitual_km
    - 0.00035 * idade_conta_dias
    + 0.55 * falhas_login_24h
    + 0.85 * compra_internacional
    + 1.15 * dispositivo_novo
    + 0.45 * horario_noturno
    + 1.40 * historico_chargeback
    + 1.55 * limite_utilizado_pct
)

prob_fraude = 1 / (
    1 + np.exp(-logit)
)

fraude = rng.binomial(
    1,
    prob_fraude
)

df = pd.DataFrame({
    "valor_transacao": valor_transacao.round(2),
    "transacoes_1h": transacoes_1h,
    "distancia_habitual_km": distancia_habitual_km.round(2),
    "idade_conta_dias": idade_conta_dias.round(0),
    "falhas_login_24h": falhas_login_24h,
    "compra_internacional": compra_internacional,
    "dispositivo_novo": dispositivo_novo,
    "horario_noturno": horario_noturno,
    "historico_chargeback": historico_chargeback,
    "limite_utilizado_pct": limite_utilizado_pct.round(3),
    "fraude": fraude
})

df.head()

In [ ]:
# Observa a frequência das classes.
distribuicao = pd.DataFrame({
    "quantidade": (
        df["fraude"]
        .value_counts()
        .sort_index()
    ),
    "percentual": (
        df["fraude"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
})

distribuicao.index = [
    "legítima",
    "fraude"
]

distribuicao

### Pergunta 1

A classe positiva é majoritária ou minoritária?

Antes de falar em explicabilidade, ainda precisamos de um modelo preditivo. Entretanto, nesta aula o foco não será otimizar desempenho, e sim **compreender suas decisões**.

# 7. Treinando um modelo para explicar

Utilizaremos uma **Random Forest**.

Esse modelo é adequado para a aula porque:

- captura relações não lineares;
- combina muitas árvores;
- possui boa capacidade preditiva;
- é menos transparente que uma única árvore;
- possui suporte eficiente no SHAP.

Assim, temos um modelo suficientemente complexo para justificar o uso de técnicas de XAI.

In [ ]:
# Atributos utilizados pelo modelo.
atributos = [
    "valor_transacao",
    "transacoes_1h",
    "distancia_habitual_km",
    "idade_conta_dias",
    "falhas_login_24h",
    "compra_internacional",
    "dispositivo_novo",
    "horario_noturno",
    "historico_chargeback",
    "limite_utilizado_pct"
]

X = df[atributos]
y = df["fraude"]

# Separa treino e teste.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE
)

# Cria a Random Forest.
modelo = RandomForestClassifier(
    n_estimators=250,
    max_depth=8,
    min_samples_leaf=8,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Treina o modelo.
modelo.fit(
    X_train,
    y_train
)

# Previsões.
pred = modelo.predict(
    X_test
)

prob = modelo.predict_proba(
    X_test
)[:, 1]

print(
    "Acurácia:",
    round(
        accuracy_score(
            y_test,
            pred
        ),
        3
    )
)

print(
    "Precisão:",
    round(
        precision_score(
            y_test,
            pred,
            zero_division=0
        ),
        3
    )
)

print(
    "Recall:",
    round(
        recall_score(
            y_test,
            pred,
            zero_division=0
        ),
        3
    )
)

print(
    "F1:",
    round(
        f1_score(
            y_test,
            pred,
            zero_division=0
        ),
        3
    )
)

O modelo possui desempenho suficiente para nossa finalidade didática.

Agora deixaremos de perguntar apenas:

> “O modelo acertou?”

e passaremos a perguntar:

> **“Quais atributos contribuíram para a decisão?”**

# 8. Antes de SHAP: importância nativa da Random Forest

A Random Forest já possui uma medida interna de importância dos atributos.

Vamos observá-la antes de utilizar XAI.

In [ ]:
importancias_rf = pd.Series(
    modelo.feature_importances_,
    index=atributos
).sort_values(
    ascending=False
)

importancias_rf.to_frame(
    "importancia"
).round(3)

In [ ]:
importancias_rf.sort_values().plot(
    kind="barh",
    figsize=(8, 5)
)

plt.xlabel("Importância")
plt.ylabel("Atributo")
plt.title(
    "Importância nativa da Random Forest"
)
plt.tight_layout()
plt.show()

Essa medida indica quais atributos foram úteis nas divisões das árvores.

Mas ela não responde diretamente:

- se um atributo aumentou ou diminuiu uma previsão;
- por que um cliente específico recebeu determinada probabilidade;
- como as contribuições se distribuem entre as observações.

É aqui que métodos como SHAP se tornam úteis.

# 9. SHAP na prática

SHAP é baseado em ideias da teoria dos jogos.

De forma simplificada, ele distribui a diferença entre uma previsão e uma referência entre os atributos utilizados pelo modelo.

Uma representação aditiva é:

$$
f(x) =
\text{valor de referência}
+
\sum_{j=1}^{p}\phi_j
$$

onde:

- $f(x)$ é a saída do modelo para a observação;
- $\phi_j$ é a contribuição SHAP do atributo $j$;
- contribuições positivas empurram a previsão em uma direção;
- contribuições negativas empurram na direção oposta.

No nosso problema, analisaremos principalmente a saída associada à classe **fraude**.

## 9.1 Criando o explicador SHAP

Como estamos usando um modelo baseado em árvores, utilizaremos `TreeExplainer`.

In [ ]:
# Cria um explicador especializado em modelos de árvore.
explicador_shap = shap.TreeExplainer(
    modelo
)

# Para manter a aula rápida, explicaremos uma amostra
# de 500 observações do teste.
X_shap = X_test.sample(
    n=min(
        500,
        len(X_test)
    ),
    random_state=RANDOM_STATE
)

# Calcula as explicações.
explicacoes_brutas = explicador_shap(
    X_shap
)

print(
    "Formato dos valores SHAP:",
    explicacoes_brutas.values.shape
)

Para classificadores binários baseados em Random Forest, versões atuais do SHAP podem retornar uma dimensão adicional correspondente às classes.

A próxima célula seleciona explicitamente a explicação da **classe 1 = fraude**.

In [ ]:
# Caso a explicação possua três dimensões:
# observação × atributo × classe
if explicacoes_brutas.values.ndim == 3:

    explicacoes_fraude = shap.Explanation(
        values=explicacoes_brutas.values[:, :, 1],
        base_values=explicacoes_brutas.base_values[:, 1],
        data=explicacoes_brutas.data,
        feature_names=atributos
    )

# Algumas versões/modelos podem retornar diretamente
# observação × atributo.
else:
    explicacoes_fraude = explicacoes_brutas

print(
    "Formato utilizado na análise:",
    explicacoes_fraude.values.shape
)

# 10. SHAP global: quais atributos mais influenciam o modelo?

Uma forma comum de obter importância global com SHAP é calcular a média do **valor absoluto** das contribuições.

Usamos valor absoluto porque queremos saber a intensidade da contribuição, independentemente de ela aumentar ou diminuir a previsão.

In [ ]:
importancia_shap = pd.Series(
    np.abs(
        explicacoes_fraude.values
    ).mean(axis=0),
    index=atributos
).sort_values(
    ascending=False
)

importancia_shap.to_frame(
    "media_abs_shap"
).round(4)

In [ ]:
# Gráfico global nativo do SHAP.
shap.plots.bar(
    explicacoes_fraude,
    max_display=10,
    show=True
)

### Pergunta 2

Compare a importância nativa da Random Forest com a importância global pelo SHAP.

Os rankings são exatamente iguais?

Eles não precisam ser.

As duas medidas respondem a perguntas relacionadas, mas são calculadas de maneiras diferentes.

# 11. SHAP beeswarm: importância e direção

O gráfico `beeswarm` fornece mais informação que um simples ranking.

Cada ponto representa uma observação.

A posição horizontal representa a contribuição SHAP do atributo para a previsão.

Assim, conseguimos observar:

- intensidade;
- direção;
- distribuição das contribuições.

In [ ]:
shap.plots.beeswarm(
    explicacoes_fraude,
    max_display=10,
    show=True
)

### Como ler o beeswarm

De forma geral:

- valores SHAP mais à direita aumentam a saída associada à fraude;
- valores mais à esquerda diminuem essa saída;
- a cor representa o valor original do atributo.

### Pergunta 3

Observe `historico_chargeback`, `dispositivo_novo` e `idade_conta_dias`.

Os valores maiores desses atributos empurram a previsão sempre na mesma direção?

Que padrões você consegue identificar?

# 12. Relação entre valor do atributo e contribuição SHAP

Também podemos analisar um atributo individualmente.

Vamos observar como o valor de `distancia_habitual_km` se relaciona com sua contribuição para a previsão.

In [ ]:
shap.plots.scatter(
    explicacoes_fraude[
        :,
        "distancia_habitual_km"
    ],
    show=True
)

Esse gráfico não é simplesmente uma correlação entre o atributo e a classe.

Ele mostra como os diferentes valores desse atributo estão associados à **contribuição produzida pelo modelo**.

# 13. Explicação local com SHAP

Agora explicaremos uma única transação.

Escolheremos uma observação que recebeu probabilidade relativamente alta de fraude.

In [ ]:
# Organiza probabilidades do conjunto de teste.
analise_teste = X_test.copy()

analise_teste["classe_real"] = (
    y_test
)

analise_teste["prob_fraude"] = (
    prob
)

# Ordena pelas maiores probabilidades.
analise_teste.sort_values(
    "prob_fraude",
    ascending=False
).head(10)

In [ ]:
# Seleciona a posição da maior probabilidade.
posicao_alta = int(
    np.argmax(
        prob
    )
)

# Recupera a observação.
transacao_alta = X_test.iloc[
    posicao_alta
]

print(
    "Probabilidade prevista de fraude:",
    round(
        prob[posicao_alta],
        3
    )
)

print(
    "Classe real:",
    int(
        y_test.iloc[
            posicao_alta
        ]
    )
)

display(
    transacao_alta.to_frame(
        "valor"
    )
)

## 13.1 Calculando SHAP para a transação

Explicaremos exatamente essa observação, e não apenas a amostra usada nos gráficos globais.

In [ ]:
# Mantém o formato DataFrame com uma linha.
X_instancia = X_test.iloc[
    [posicao_alta]
]

explicacao_instancia_bruta = (
    explicador_shap(
        X_instancia
    )
)

if (
    explicacao_instancia_bruta
    .values
    .ndim
    == 3
):

    explicacao_instancia = shap.Explanation(
        values=(
            explicacao_instancia_bruta
            .values[0, :, 1]
        ),
        base_values=(
            explicacao_instancia_bruta
            .base_values[0, 1]
        ),
        data=(
            explicacao_instancia_bruta
            .data[0]
        ),
        feature_names=atributos
    )

else:

    explicacao_instancia = (
        explicacao_instancia_bruta[0]
    )

In [ ]:
# Waterfall: parte do valor de referência
# e mostra como cada atributo desloca a previsão.
shap.plots.waterfall(
    explicacao_instancia,
    max_display=10,
    show=True
)

### Como interpretar o waterfall

A explicação começa em um **valor de referência**.

Cada atributo desloca a saída:

- em direção a maior risco;
- ou em direção a menor risco.

A soma dessas contribuições leva à saída final do modelo para aquela observação.

### Pergunta 4

Quais três atributos mais contribuíram para aumentar a previsão de fraude?

Quais atributos contribuíram para reduzi-la?

# 14. Verificando a propriedade aditiva do SHAP

Vamos conferir numericamente:

```text
valor de referência + soma das contribuições SHAP
```

Para esse modelo, a soma deve reproduzir aproximadamente a saída explicada para a classe fraude.

In [ ]:
valor_base = float(
    explicacao_instancia
    .base_values
)

soma_contribuicoes = float(
    explicacao_instancia
    .values
    .sum()
)

saida_reconstruida = (
    valor_base
    + soma_contribuicoes
)

prob_modelo = modelo.predict_proba(
    X_test.iloc[
        [posicao_alta]
    ]
)[0, 1]

print(
    "Valor de referência:",
    round(
        valor_base,
        4
    )
)

print(
    "Soma das contribuições:",
    round(
        soma_contribuicoes,
        4
    )
)

print(
    "Referência + contribuições:",
    round(
        saida_reconstruida,
        4
    )
)

print(
    "Probabilidade do modelo:",
    round(
        prob_modelo,
        4
    )
)

# 15. Comparando com uma transação de baixo risco

Interpretabilidade local significa que duas observações podem receber explicações muito diferentes.

Vamos selecionar agora uma transação com baixa probabilidade de fraude.

In [ ]:
posicao_baixa = int(
    np.argmin(
        prob
    )
)

print(
    "Probabilidade prevista:",
    round(
        prob[posicao_baixa],
        4
    )
)

display(
    X_test.iloc[
        posicao_baixa
    ].to_frame(
        "valor"
    )
)

In [ ]:
X_baixa = X_test.iloc[
    [posicao_baixa]
]

exp_baixa_bruta = (
    explicador_shap(
        X_baixa
    )
)

if (
    exp_baixa_bruta
    .values
    .ndim
    == 3
):

    exp_baixa = shap.Explanation(
        values=(
            exp_baixa_bruta
            .values[0, :, 1]
        ),
        base_values=(
            exp_baixa_bruta
            .base_values[0, 1]
        ),
        data=(
            exp_baixa_bruta
            .data[0]
        ),
        feature_names=atributos
    )

else:
    exp_baixa = (
        exp_baixa_bruta[0]
    )

shap.plots.waterfall(
    exp_baixa,
    max_display=10,
    show=True
)

### Pergunta 5

Compare as duas explicações locais.

Os mesmos atributos aparecem com a mesma intensidade e direção?

Essa comparação ajuda a perceber uma ideia importante:

> uma importância global elevada não significa que o atributo será decisivo em todas as observações.

# 16. LIME na prática

LIME significa **Local Interpretable Model-agnostic Explanations**.

A ideia é diferente da utilizada pelo SHAP.

Para explicar uma observação, o LIME:

1. cria pequenas perturbações ao redor da observação;
2. consulta o modelo original para obter previsões nessas perturbações;
3. dá maior peso às observações próximas do exemplo original;
4. ajusta um modelo simples localmente;
5. utiliza esse modelo simples como aproximação da decisão naquele entorno.

Portanto:

> o LIME não tenta reproduzir o modelo inteiro. Ele tenta criar uma aproximação interpretável **na vizinhança da observação explicada**.

## 16.1 Criando o explicador LIME

Utilizaremos a mesma Random Forest.

Isso ilustra uma característica importante do LIME: ele é **agnóstico ao modelo**.

O explicador precisa conseguir consultar uma função que retorne probabilidades.

In [ ]:
explicador_lime = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=atributos,
    class_names=[
        "legítima",
        "fraude"
    ],
    mode="classification",
    discretize_continuous=True,
    random_state=RANDOM_STATE
)

print(
    "Explicador LIME criado."
)

# 17. Explicando a mesma transação com LIME

Usaremos a mesma transação de alta probabilidade explicada anteriormente pelo SHAP.

Assim poderemos comparar os métodos.

In [ ]:
explicacao_lime = (
    explicador_lime
    .explain_instance(
        data_row=(
            X_test.iloc[
                posicao_alta
            ].values
        ),
        predict_fn=(
            modelo.predict_proba
        ),
        num_features=10,
        labels=(1,)
    )
)

# A lista contém condições locais
# e o peso atribuído a cada uma.
explicacao_lime.as_list(
    label=1
)

In [ ]:
# Converte a explicação para um gráfico matplotlib.
figura = (
    explicacao_lime
    .as_pyplot_figure(
        label=1
    )
)

plt.title(
    "LIME — explicação local da transação"
)

plt.tight_layout()
plt.show()

### Como interpretar o LIME

O LIME apresenta condições como:

```text
dispositivo_novo > 0,5
```

ou intervalos de valores.

Os pesos indicam como essas condições contribuem **no modelo local aproximado** construído ao redor da observação.

Os valores não devem ser comparados numericamente de forma direta com valores SHAP, pois os métodos possuem fundamentos diferentes.

## 17.1 O modelo original e a aproximação local

É importante lembrar que LIME explica um modelo complexo usando um modelo simples local.

A explicação é uma **aproximação**.

Vamos exibir a probabilidade original e o score da explicação local quando disponível.

In [ ]:
print(
    "Probabilidade original de fraude:",
    round(
        modelo.predict_proba(
            X_test.iloc[
                [posicao_alta]
            ]
        )[0, 1],
        4
    )
)

print(
    "Qualidade local da aproximação (R²):",
    round(
        explicacao_lime.score,
        4
    )
)

Um score local maior indica que o modelo linear utilizado pelo LIME reproduziu melhor o comportamento do modelo original naquela vizinhança.

Isso não significa que a explicação seja perfeita nem que possa ser generalizada para toda a base.

# 18. SHAP × LIME

Os dois métodos explicam modelos, mas não funcionam da mesma maneira.

| Aspecto | SHAP | LIME |
|---|---|---|
| Base conceitual | valores de Shapley | aproximação local |
| Explicação local | sim | sim |
| Análise global | sim, agregando explicações | não é seu foco principal |
| Dependência do modelo | possui variantes específicas e agnósticas | agnóstico |
| Resultado | contribuição dos atributos | pesos do modelo local |
| Sensibilidade local | depende do método/configuração | pode variar com amostragem e vizinhança |

Não devemos esperar que os dois produzam rankings idênticos.

### Pergunta 6

Compare a explicação SHAP e a explicação LIME da transação de alta probabilidade.

**a)** Quais atributos aparecem como mais relevantes em ambos?

**b)** Existe algum atributo que aparece forte em um método e menos destacado no outro?

**c)** Isso significa necessariamente que um dos métodos está errado?

# 19. Estabilidade das explicações locais

Como o LIME utiliza amostras perturbadas ao redor da observação, pequenas alterações na configuração ou na amostragem podem modificar a explicação.

Vamos criar outro explicador com uma semente diferente.

In [ ]:
explicador_lime_2 = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=atributos,
    class_names=[
        "legítima",
        "fraude"
    ],
    mode="classification",
    discretize_continuous=True,
    random_state=99
)

explicacao_lime_2 = (
    explicador_lime_2
    .explain_instance(
        data_row=(
            X_test.iloc[
                posicao_alta
            ].values
        ),
        predict_fn=(
            modelo.predict_proba
        ),
        num_features=10,
        labels=(1,)
    )
)

comparacao_lime = pd.DataFrame({
    "LIME_seed_42": dict(
        explicacao_lime.as_list(
            label=1
        )
    ),
    "LIME_seed_99": dict(
        explicacao_lime_2.as_list(
            label=1
        )
    )
})

comparacao_lime

### Pergunta 7

As duas explicações LIME são exatamente iguais?

Se houver diferenças, elas demonstram uma característica importante das explicações locais:

> interpretabilidade também precisa ser analisada quanto à **estabilidade**.

# 20. Explicando um erro do modelo

Uma aplicação interessante de XAI é investigar previsões incorretas.

Vamos procurar uma observação que o modelo classificou incorretamente.

In [ ]:
# Identifica as posições em que previsão e classe real diferem.
posicoes_erros = np.where(
    pred != y_test.to_numpy()
)[0]

print(
    "Quantidade de erros no teste:",
    len(posicoes_erros)
)

if len(posicoes_erros) > 0:

    posicao_erro = int(
        posicoes_erros[0]
    )

    print(
        "Classe real:",
        int(
            y_test.iloc[
                posicao_erro
            ]
        )
    )

    print(
        "Classe prevista:",
        int(
            pred[
                posicao_erro
            ]
        )
    )

    print(
        "Probabilidade de fraude:",
        round(
            prob[
                posicao_erro
            ],
            3
        )
    )

    display(
        X_test.iloc[
            posicao_erro
        ].to_frame(
            "valor"
        )
    )

In [ ]:
X_erro = X_test.iloc[
    [posicao_erro]
]

exp_erro_bruta = (
    explicador_shap(
        X_erro
    )
)

if (
    exp_erro_bruta
    .values
    .ndim
    == 3
):

    exp_erro = shap.Explanation(
        values=(
            exp_erro_bruta
            .values[0, :, 1]
        ),
        base_values=(
            exp_erro_bruta
            .base_values[0, 1]
        ),
        data=(
            exp_erro_bruta
            .data[0]
        ),
        feature_names=atributos
    )

else:
    exp_erro = (
        exp_erro_bruta[0]
    )

shap.plots.waterfall(
    exp_erro,
    max_display=10,
    show=True
)

### Pergunta 8

Analise o erro.

A explicação ajuda a compreender **por que o modelo chegou à decisão incorreta**?

Observe que explicar um erro não o transforma em acerto.

A técnica apenas ajuda a investigar o comportamento do modelo.

# 21. O que SHAP e LIME NÃO demonstram

Este ponto é fundamental.

Se SHAP indicar:

```text
dispositivo_novo aumentou a previsão de fraude
```

isso significa:

> para esse modelo, considerando os dados utilizados, esse atributo contribuiu para aumentar a saída.

Isso **não significa automaticamente**:

> utilizar um dispositivo novo causa fraude.

Da mesma forma, uma explicação não prova:

- causalidade;
- ausência de viés;
- qualidade dos dados;
- justiça do modelo;
- validade fora da população analisada;
- correção da previsão.

XAI ajuda a investigar um modelo. Não substitui validação científica, análise de domínio e avaliação ética.

# 22. Exercícios

## Exercício 1 — importância global com SHAP

Utilize os valores SHAP já calculados.

Crie uma tabela com:

- nome do atributo;
- média do valor SHAP absoluto;
- posição no ranking.

Depois compare os cinco primeiros atributos com `feature_importances_` da Random Forest.

In [ ]:
# Complete aqui.

## Exercício 2 — outra explicação local

Escolha uma transação do conjunto de teste com:

```python
0.40 <= probabilidade de fraude <= 0.60
```

Explique-a com SHAP.

Responda:

1. quais atributos aumentam o risco?
2. quais diminuem?
3. a decisão parece estar próxima do valor de referência?

In [ ]:
# Complete aqui.

## Exercício 3 — LIME

Explique a mesma observação do Exercício 2 utilizando LIME.

Compare:

- os três atributos mais importantes pelo SHAP;
- os três atributos mais importantes pelo LIME.

Os resultados são idênticos?

In [ ]:
# Complete aqui.

## Exercício 4 — interpretação crítica

Escolha uma afirmação e explique por que ela está incorreta:

**A)** “O SHAP mostrou que dispositivo novo causa fraude.”

**B)** “O LIME explica perfeitamente o comportamento global da Random Forest.”

**C)** “Se SHAP e LIME discordarem, obrigatoriamente um deles está errado.”

**D)** “Um modelo explicável é automaticamente um modelo justo.”

# 23. Desafio final — relatório de explicabilidade

Imagine que a área de prevenção a fraudes questionou uma transação bloqueada.

Você deve produzir uma explicação curta.

Preencha:

**Probabilidade prevista de fraude:** __________________________

**Três atributos que mais aumentaram a previsão segundo SHAP:**

1. __________________________________________________________
2. __________________________________________________________
3. __________________________________________________________

**Dois atributos que reduziram a previsão:**

1. __________________________________________________________
2. __________________________________________________________

**Principais condições destacadas pelo LIME:**  
_____________________________________________________________

**SHAP e LIME apresentaram interpretação semelhante?**  
_____________________________________________________________

**A explicação demonstra que esses atributos causaram fraude?**  
_____________________________________________________________

**Que informação adicional você solicitaria antes de tomar uma decisão real?**  
_____________________________________________________________

# 24. Resumo: quando usar cada abordagem?

**Importância nativa do modelo**

Útil para uma visão inicial, mas geralmente limitada.

**SHAP global**

Útil para entender quais atributos mais influenciam o modelo em muitas observações.

**SHAP local**

Útil para decompor uma previsão específica em contribuições dos atributos.

**LIME**

Útil para construir uma aproximação interpretável na vizinhança de uma observação.

Nenhum método deve ser utilizado de forma automática ou isolada.

# 25. Reflexão final

Responda brevemente:

**1. Qual é a diferença entre interpretabilidade global e local?**

**2. O que representa um valor SHAP positivo para a classe fraude?**

**3. Por que utilizamos o valor absoluto dos SHAP values para calcular importância global?**

**4. O que representa o valor de referência em uma explicação SHAP?**

**5. Qual é a ideia central do LIME?**

**6. Por que LIME é chamado de model-agnostic?**

**7. Por que duas execuções do LIME podem produzir explicações ligeiramente diferentes?**

**8. Uma explicação SHAP demonstra causalidade?**

**9. Por que é útil explicar previsões incorretas?**

**10. Um modelo com boa explicabilidade é necessariamente um bom modelo?**

# 26. Conclusão

Nesta aula percorremos:

**modelo preditivo → importância global → SHAP global → SHAP local → LIME → comparação → análise crítica**

A principal ideia é:

> **interpretabilidade procura tornar o comportamento do modelo mais compreensível, mas não transforma automaticamente correlação em causalidade nem previsão em verdade.**

SHAP ajuda a decompor a saída do modelo em contribuições dos atributos.

LIME cria uma aproximação simples ao redor de uma observação específica.

Em Ciência de Dados, explicações são úteis para:

- investigar decisões;
- detectar comportamentos inesperados;
- comunicar resultados;
- analisar erros;
- aumentar a auditabilidade.

Mas devem sempre ser interpretadas junto com:

- qualidade dos dados;
- desempenho do modelo;
- conhecimento do domínio;
- limitações metodológicas;
- contexto da decisão.